#Coleta do corpus

##Conexão com Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Instalação da biblioteca google-play-scraper para coleta das avaliações

In [ ]:
%pip install google-play-scraper==1.2.7

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00


## Configuração da coleta

Serão consultados dez assistentes de IA generativa, com uma meta
de 100 avaliações válidas por nota e aplicativo.

As consultas utilizarão português, região brasileira e ordenação
pelas avaliações mais relevantes.

In [ ]:
from google_play_scraper import Sort
from pathlib import Path

APLICATIVOS = {
    "ChatGPT": "com.openai.chatgpt",
    "Google Gemini": "com.google.android.apps.bard",
    "Grok": "ai.x.grok",
    "Meta AI": "com.facebook.stella",
    "Perplexity": "ai.perplexity.app.android",
    "Nova": "com.scaleup.chatai",
    "Dola": "com.larus.wolf",
    "Claude": "com.anthropic.claude",
    "DeepSeek": "com.deepseek.chat",
    "Luzia": "co.thewordlab.luzia",
}

COTAS_POR_NOTA = {
    1: 100,
    2: 100,
    3: 100,
    4: 100,
    5: 100,
}

IDIOMA = "pt"
PAIS = "br"
ORDENACAO = Sort.MOST_RELEVANT

PASTA_RAW = Path("/content/drive/MyDrive/pln_trabalho/data/raw")

if not PASTA_RAW.is_dir():
    raise FileNotFoundError(
        "Pasta data/raw não encontrada. Confira a conexão com o Drive "
        "e o caminho da pasta pln_trabalho."
    )

meta_por_app = sum(COTAS_POR_NOTA.values())
meta_total = len(APLICATIVOS) * meta_por_app

print(f"Aplicativos: {len(APLICATIVOS)}")
print(f"Meta por aplicativo: {meta_por_app}")
print(f"Meta total: {meta_total}")
print(f"Destino dos dados brutos: {PASTA_RAW}")

Aplicativos: 10
Meta por aplicativo: 500
Meta total: 5000
Destino dos dados brutos: /content/drive/MyDrive/pln_trabalho/data/raw


## Coleta dos dados brutos

Coleta de avaliações por aplicativo e nota filtradas por relevancia

Cada consulta é salva separadamente em JSON, com os parâmetros
utilizados.

Os textos são preservados sem alterações.

In [ ]:
import json
import time
from datetime import datetime, timezone
from importlib.metadata import version

from google_play_scraper import reviews

CAMPOS = [
    "reviewId",
    "content",
    "score",
    "thumbsUpCount",
    "reviewCreatedVersion",
    "at",
]

PAUSA_SEGUNDOS = 3


def converter_data(valor):
    if isinstance(valor, datetime):
        return valor.isoformat()
    raise TypeError(f"Tipo não suportado: {type(valor).__name__}")


def coletar_avaliacoes():
    if not PASTA_RAW.is_dir():
        raise FileNotFoundError(
            f"Pasta não encontrada: {PASTA_RAW}. Confira a conexão com o Drive."
        )

    total_salvo = 0
    consultas_abaixo_da_meta = []

    for nome_app, app_id in APLICATIVOS.items():
        print(f"\nAplicativo: {nome_app}", flush=True)

        for nota, quantidade in COTAS_POR_NOTA.items():
            arquivo = PASTA_RAW / f"{app_id}_nota_{nota}.json"

            if arquivo.exists():
                # Lê o arquivo para conferir o resultado já preservado.
                with arquivo.open("r", encoding="utf-8") as entrada:
                    pacote = json.load(entrada)

                parametros = pacote["parametros"]

                if (
                    parametros["app_id"] != app_id
                    or parametros["nota"] != nota
                    or parametros["quantidade_solicitada"] != quantidade
                    or parametros["idioma"] != IDIOMA
                    or parametros["pais"] != PAIS
                    or parametros["ordenacao"] != ORDENACAO.name
                ):
                    raise ValueError(
                        f"Configuração diferente da registrada em {arquivo.name}"
                    )

                recebidas = len(pacote["avaliacoes"])
                print(
                    f"  Nota {nota}: arquivo existente "
                    f"({recebidas}/{quantidade}).",
                    flush=True,
                )

            else:
                print(
                    f"  Nota {nota}: solicitando {quantidade} avaliações...",
                    flush=True,
                )

                try:
                    resultado, _ = reviews(
                        app_id,
                        lang=IDIOMA,
                        country=PAIS,
                        sort=ORDENACAO,
                        count=quantidade,
                        filter_score_with=nota,
                    )
                except Exception as erro:
                    raise RuntimeError(
                        f"Falha em {nome_app}, nota {nota}. "
                        "Os arquivos anteriores foram preservados."
                    ) from erro

                coletado_em = datetime.now(timezone.utc).isoformat()

                avaliacoes = [
                    {
                        **{campo: item.get(campo) for campo in CAMPOS},
                        "app_name": nome_app,
                        "app_id": app_id,
                        "collected_at": coletado_em,
                    }
                    for item in resultado
                ]

                pacote = {
                    "parametros": {
                        "app_name": nome_app,
                        "app_id": app_id,
                        "nota": nota,
                        "quantidade_solicitada": quantidade,
                        "idioma": IDIOMA,
                        "pais": PAIS,
                        "ordenacao": ORDENACAO.name,
                        "biblioteca": "google-play-scraper",
                        "versao": version("google-play-scraper"),
                    },
                    "collected_at": coletado_em,
                    "quantidade_recebida": len(avaliacoes),
                    "avaliacoes": avaliacoes,
                }

                # Finaliza o arquivo apenas depois de concluir a escrita.
                temporario = arquivo.with_suffix(".json.tmp")

                with temporario.open("w", encoding="utf-8") as saida:
                    json.dump(
                        pacote,
                        saida,
                        ensure_ascii=False,
                        indent=2,
                        default=converter_data,
                    )

                temporario.replace(arquivo)

                recebidas = len(avaliacoes)
                print(
                    f"  Nota {nota}: {recebidas}/{quantidade} salvas.",
                    flush=True,
                )

                time.sleep(PAUSA_SEGUNDOS)

            total_salvo += recebidas

            if recebidas < quantidade:
                consultas_abaixo_da_meta.append(
                    (nome_app, nota, recebidas, quantidade)
                )

    print(f"\nTotal de registros nos arquivos: {total_salvo}")
    print(f"Destino: {PASTA_RAW}")

    if consultas_abaixo_da_meta:
        print("\nConsultas abaixo da meta:")
        for nome, nota, recebidas, solicitadas in consultas_abaixo_da_meta:
            print(f"- {nome}, nota {nota}: {recebidas}/{solicitadas}")
    else:
        print("Todas as consultas atingiram a quantidade solicitada.")


coletar_avaliacoes()


Aplicativo: ChatGPT
  Nota 1: solicitando 100 avaliações...
  Nota 1: 100/100 salvas.
  Nota 2: solicitando 100 avaliações...
  Nota 2: 100/100 salvas.
  Nota 3: solicitando 100 avaliações...
  Nota 3: 100/100 salvas.
  Nota 4: solicitando 100 avaliações...
  Nota 4: 100/100 salvas.
  Nota 5: solicitando 100 avaliações...
  Nota 5: 100/100 salvas.

Aplicativo: Google Gemini
  Nota 1: solicitando 100 avaliações...
  Nota 1: 100/100 salvas.
  Nota 2: solicitando 100 avaliações...
  Nota 2: 100/100 salvas.
  Nota 3: solicitando 100 avaliações...
  Nota 3: 100/100 salvas.
  Nota 4: solicitando 100 avaliações...
  Nota 4: 100/100 salvas.
  Nota 5: solicitando 100 avaliações...
  Nota 5: 100/100 salvas.

Aplicativo: Grok
  Nota 1: solicitando 100 avaliações...
  Nota 1: 100/100 salvas.
  Nota 2: solicitando 100 avaliações...
  Nota 2: 100/100 salvas.
  Nota 3: solicitando 100 avaliações...
  Nota 3: 100/100 salvas.
  Nota 4: solicitando 100 avaliações...
  Nota 4: 100/100 salvas.
  Nota 5: 

###Conferência dos dados coletados
Verificação da quantidade de avaliações por aplicativo e nota, de IDs ausentes ou repetidos, de textos vazios e da correspondência das notas e dos aplicativos com os parâmetros da coleta.

In [ ]:
import json
from collections import Counter

diagnostico = Counter({
    "Arquivos conferidos": 0,
    "Avaliações conferidas": 0,
    "Arquivos com quantidade diferente da meta": 0,
    "IDs ausentes ou inválidos": 0,
    "IDs repetidos no mesmo aplicativo": 0,
    "Textos vazios ou inválidos": 0,
    "Notas inválidas ou diferentes do filtro": 0,
    "Aplicativos divergentes": 0,
})
ids_vistos = set()

for nome_app, app_id in APLICATIVOS.items():
    for nota, meta in COTAS_POR_NOTA.items():
        arquivo = PASTA_RAW / f"{app_id}_nota_{nota}.json"

        with arquivo.open(encoding="utf-8") as entrada:
            avaliacoes = json.load(entrada)["avaliacoes"]

        diagnostico["Arquivos conferidos"] += 1
        diagnostico["Arquivos com quantidade diferente da meta"] += (
            len(avaliacoes) != meta
        )

        for avaliacao in avaliacoes:
            diagnostico["Avaliações conferidas"] += 1

            review_id = avaliacao.get("reviewId")
            texto = avaliacao.get("content")
            score = avaliacao.get("score")

            if not isinstance(review_id, str) or not review_id.strip():
                diagnostico["IDs ausentes ou inválidos"] += 1
            else:
                chave = (app_id, review_id)
                diagnostico["IDs repetidos no mesmo aplicativo"] += (
                    chave in ids_vistos
                )
                ids_vistos.add(chave)

            diagnostico["Textos vazios ou inválidos"] += (
                not isinstance(texto, str) or not texto.strip()
            )
            diagnostico["Notas inválidas ou diferentes do filtro"] += (
                type(score) is not int or score != nota
            )
            diagnostico["Aplicativos divergentes"] += (
                avaliacao.get("app_id") != app_id
                or avaliacao.get("app_name") != nome_app
            )

for descricao, quantidade in diagnostico.items():
    print(f"{descricao}: {quantidade}")


Arquivos conferidos: 50
Avaliações conferidas: 5000
Arquivos com quantidade diferente da meta: 0
IDs ausentes ou inválidos: 0
IDs repetidos no mesmo aplicativo: 0
Textos vazios ou inválidos: 0
Notas inválidas ou diferentes do filtro: 0
Aplicativos divergentes: 0
